In [ ]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
import torch, rdkit
import os
from pathlib import Path
import sys
def _is_models_root(p: Path) -> bool:
    return (p / "utils").is_dir() and (p / "models").is_dir() and (p / "notebooks").is_dir() and (p / "data").is_dir()


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if _is_models_root(p):
            return p
        cand = p / "MY_PAPER_RELATED" / "MODELS"
        if _is_models_root(cand):
            return cand
    raise FileNotFoundError("Could not locate MODELS project root")
PROJECT_ROOT = resolve_project_root()
os.environ["MODELS_VARIANT"] = "TransCVAE_PSMILES"
os.environ["MODELS_FULL_DATA_TRAINING"] = "1"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rdkit import Chem
from utils.utils import *
from utils.dataloader import *
from utils.loss_fn import *
from utils.eval import *

from tqdm import trange
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_ps = PS("CC(CCCNC(=O)C(C)OC(=O)[*])O[*]").canonicalize.psmiles
print(test_ps)
print(split_psmiles_tokens(test_ps))
vocab = dataset.vocab
index_to_token = {idx: token for token, idx in vocab.items()}
print("Split summary:")
print(split_summary.to_string(index=False))
print("Split leakage counts:", split_info["leak_counts"])
print(f"Split tag: {SPLIT_TAG}")
print(f"Full-data training mode: {FULL_DATA_TRAINING}")
print(f"Dataset sizes - train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")


In [ ]:
def select_model(latent):
    from models.Trans import CVAE, PriorNet
    model = CVAE(latent_dim=latent).to(device)
    prior = PriorNet(y_dim=1, latent_dim=latent).to(device)
    return model, prior


def get_loss_fn(latent):
    # Validity-first rebalance for PSMILES: reduce invalid generations while keeping diversity.
    loss_fn = ConditionalVAELoss(
        vocab_size=dataset.vocab_size,
        max_beta=0.72,
        cyc_steps=1200,
        num_cycles=1,
        anneal_steps=600,
        free_bits=0.05,
        capacity_max=0.0,
        capacity_inc=0.0,
        gamma=0.8,
        prop_w=0.12,
        nce=0.001,
        sig_pen_p=0.0022,
        sig_pen_q=0.0024,
        imb=0.04,
        mu_align_w=0.0007,
        sigma_p_target=0.210,
        sigma_q_target=0.195,
        sigma_p_target_w=0.16,
        sigma_q_target_w=0.07,
        sigma_align_w=0.022,
        sigma_p_var_floor=0.048,
        sigma_p_var_w=0.95,
        sigma_q_var_floor=0.017,
        sigma_q_var_w=0.08,
        sigma_p_upper_margin=0.14,
        sigma_p_upper_w=0.12,
        sigma_p_hi_ratio_target=0.48,
        sigma_p_hi_ratio_tau=0.03,
        sigma_p_hi_ratio_w=0.04,
        mu_std_floor=0.16,
        mu_std_w=0.26,
        mu_std_global_floor=0.19,
        mu_std_global_w=0.22,
        mu_energy_floor=0.036,
        mu_energy_w=0.20,
        reg_warmup_steps=750,
        latent_dim=latent,
    ).to(device)
    return loss_fn



In [ ]:
mode = "Trans"
latent_dim = 128
seq_len = dataset.max_len + 2
model, prior = select_model(latent_dim)
loss_fn = get_loss_fn(latent_dim)

for head in [model.to_prop, model.to_prop_z]:
    nn.init.xavier_uniform_(head.weight)
    nn.init.zeros_(head.bias)

prior_params = prior.parameters()

prop_params = list(model.to_prop.parameters()) + list(model.to_prop_z.parameters())
prop_params_id = {id(p) for p in prop_params}
base_params = [p for p in model.parameters() if id(p) not in prop_params_id]

lr = 3e-4
lr_prior = lr * 0.1
from torch.optim import AdamW
optim = AdamW([{'params': base_params, 'lr': lr}, {'params': prop_params, 'lr': lr * 0.5}])
optim2 = AdamW(prior_params, lr=lr_prior)


In [ ]:
import datetime
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.error')
status_out = widgets.Output()
PAD_IDX = dataset.vocab['[PAD]']
display(status_out)

epoch = 1200
model.train()
progress = tqdm(range(epoch), desc="Training")
loss_arr = []
val_loss_arr = []


def _simple_latent_mask(enc_mask):
    if mode == "LSTM":
        return torch.ones((enc_mask.size(0), 1), dtype=torch.bool, device=enc_mask.device)
    return torch.cat(
        ((~enc_mask), torch.ones(enc_mask.size(0), 1, dtype=torch.bool, device=enc_mask.device)),
        dim=1,
    )


def _forward_loss(batch, step):
    sm_enc, sm_dec_in, sm_dec_out, prop = [t.to(device) for t in batch]
    enc_mask = sm_enc == PAD_IDX
    dec_mask = sm_dec_in == PAD_IDX
    output, tgt, mu_q, lv_q, tgt_z = model(
        sm_enc,
        sm_dec_in,
        prop.float(),
        enc_mask,
        dec_mask,
    )
    mu_p, lv_p = prior(prop.float().squeeze(-1))
    latent_mask = _simple_latent_mask(enc_mask)
    loss, BCE, KLD, kld_raw, kld_per_token, props = loss_fn(
        output.float(),
        sm_dec_out,
        mu_q,
        lv_q,
        mu_p,
        lv_p,
        tgt,
        prop.float().squeeze(-1),
        tgt_z,
        step,
        latent_mask=latent_mask,
    )
    with torch.no_grad():
        pred_tok = output.argmax(dim=-1)
        nonpad_mask = sm_dec_out != PAD_IDX
        token_total = int(nonpad_mask.sum().item())
        token_correct = int(((pred_tok == sm_dec_out) & nonpad_mask).sum().item())
    return loss, BCE, KLD, kld_raw, token_total, token_correct


def _eval_loss(loader, step):
    if loader is None or len(loader) == 0:
        return float("nan"), float("nan")
    model.eval()
    prior.eval()
    batchloss = 0.0
    token_total = 0
    token_correct = 0
    with torch.no_grad():
        for batch in loader:
            loss, BCE, KLD, kld_raw, n_tok, n_correct = _forward_loss(batch, step)
            batchloss += loss.item()
            token_total += n_tok
            token_correct += n_correct
    model.train()
    return batchloss / max(1, len(loader)), token_correct / max(1, token_total)


for i in progress:
    batchloss = 0.0
    bce = 0.0
    kld = 0.0
    kld_raw_batch = 0.0
    token_total = 0
    token_correct = 0

    model.train()
    prior.train()
    for batch in train_dataloader:
        optim.zero_grad()
        optim2.zero_grad()

        loss, BCE, KLD, kld_raw, n_tok, n_correct = _forward_loss(batch, i)
        loss.backward()
        optim.step()
        optim2.step()

        batchloss += loss.item()
        bce += BCE.item()
        kld += KLD.item()
        kld_raw_batch += kld_raw.item()
        token_total += n_tok
        token_correct += n_correct

    loss = batchloss / max(1, len(train_dataloader))
    bce_epoch = bce / max(1, len(train_dataloader))
    kld_epoch = kld / max(1, len(train_dataloader))
    tok_acc = token_correct / max(1, token_total)
    loss_arr.append(loss)

    val_loss, val_tok_acc = _eval_loss(val_dataloader, i)
    val_loss_arr.append(val_loss)

    elapsed = int(progress.format_dict.get("elapsed", 0))
    formatted_elap = str(datetime.timedelta(seconds=elapsed))
    rate = progress.format_dict.get("rate", None)
    sec_per_iter = 1 / rate if rate and rate != 0 else 0
    total = int(sec_per_iter * progress.total)
    formatted_total = str(datetime.timedelta(seconds=total))

    progress.set_description(f"train={loss:.6f} val={val_loss:.6f}")
    with status_out:
        clear_output(wait=True)
        print(f"Elapsed: {formatted_elap} | sec/iter: {sec_per_iter:.3f}s")
        print("Total time:", formatted_total)
        print(f"Step: {i+1}/{progress.total}")
        print(f"train loss: {loss:.6f}, train tok_acc: {tok_acc:.4f}, BCE: {bce_epoch:.6f}, KLD: {kld_epoch:.6f}")
        print(f"val loss: {val_loss:.6f}, val tok_acc: {val_tok_acc:.4f}")
        print(f"Raw KLD: {kld_raw_batch/max(1, len(train_dataloader)):.6f}")


In [ ]:
model.eval()
prior.eval()
val_loss, val_tok_acc = _eval_loss(val_dataloader, step=0)
print(f"[VAL] loss: {val_loss:.6f}, tok_acc: {val_tok_acc:.4f}")


In [ ]:
model.eval()
prior.eval()
results = []
origin = []

properties_results = []
properties_origin = []
EVAL_SPLIT_NAME = "test"
EVAL_DATASET = test_dataset
EVAL_DATALOADER = test_dataloader
print(EVAL_SPLIT_NAME, len(EVAL_DATASET))

with torch.no_grad():
    for smiles_enc, smiles_dec_input, smiles_dec_output, properties in EVAL_DATALOADER:
        smiles_enc = smiles_enc.to(device)
        smiles_dec_input = smiles_dec_input.to(device)
        smiles_dec_output = smiles_dec_output.to(device)
        properties = properties.to(device)

        enc_smi_mask = smiles_enc == PAD_IDX
        dec_smi_mask = smiles_dec_input == PAD_IDX

        result, tgt, means, log_var, z = model(
            smiles_enc,
            smiles_dec_input,
            properties,
            enc_smi_mask,
            dec_smi_mask,
        )

        results.append(result)
        origin.append(smiles_dec_output)

        properties_results.append(tgt)
        properties_origin.append(properties.squeeze(-1))

results = torch.cat(results, dim=0)
origin = torch.cat(origin, dim=0)
results = nn.functional.softmax(results, dim=-1)
argmax_indices = torch.argmax(results, dim=-1)
output = torch.nn.functional.one_hot(argmax_indices, num_classes=results.size(-1))
print(argmax_indices)
print(results.shape)
print(origin.shape)

from sklearn.metrics import mean_absolute_error
properties_origin = torch.cat(properties_origin, dim=0)
properties_results = torch.cat(properties_results, dim=0)
MAE_2 = mean_absolute_error(properties_origin.cpu(), properties_results.cpu())
print("MAE(properties) : ", MAE_2)

results_smiles = []
origin_smiles = []

for row in argmax_indices:
    smiles = tok_ids_to_smiles(row.tolist(), index_to_token)
    results_smiles.append(smiles or "")

for row in origin:
    smiles = tok_ids_to_smiles(row.tolist(), index_to_token)
    origin_smiles.append(smiles or "")


In [ ]:
origin_smiles = [smiles.removesuffix("EOS").strip() for smiles in origin_smiles]
results_smiles = [smiles.removesuffix("EOS").strip() for smiles in results_smiles]

for i in range(len(results_smiles)):
    if(origin_smiles[i] != results_smiles[i]):
        print(i, "번째 다름!")
    print("real smiles      : ", origin_smiles[i])
    print("predicted smiles : ", results_smiles[i])


MAE = mean_absolute_error(origin.cpu(), torch.argmax(results.cpu(), dim=-1))
print("MAE : ", MAE)



In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import DataStructs, rdFingerprintGenerator
RDLogger.DisableLog('rdApp.error')

generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def tanimoto_similarity(smiles1, smiles2):
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    fp1 = generator.GetFingerprint(mol1)
    fp2 = generator.GetFingerprint(mol2)
    return DataStructs.TanimotoSimilarity(fp1, fp2)

def is_valid(smiles):
    return Chem.MolFromSmiles(smiles) is not None

TS = 0.0
canbe = 0
notbe = 0

for sm, orig in zip(results_smiles, origin_smiles):
    if(len(sm)==0):
        notbe += 1
        continue
    try:
        sm = PS(sm).canonicalize.psmiles
        if is_valid(sm) and is_valid(orig):
            sim = tanimoto_similarity(sm, orig)
            TS += sim
            canbe += 1
        else:
            notbe += 1
    except:
        notbe += 1
        continue


if canbe > 0:
    print("Tanimoto Similarity : ", TS / canbe)
else:
    print("No valid molecules to compare.")

print("가능한 분자 개수 :", canbe)
print("불가능한 분자 개수 :", notbe)
print("Valid fraction      :", canbe / len(results_smiles))


In [ ]:
def save_weights():
    save_path = PROJECT_ROOT / "checkpoints" / "model_weights_trans_TransCVAE_PSMILES.pth"
    torch.save(model.state_dict(), save_path)

    save_path = PROJECT_ROOT / "checkpoints" / "model_weights_prior_trans_TransCVAE_PSMILES.pth"
    torch.save(prior.state_dict(), save_path)


def load_weights(latent):
    from models.Trans import CVAE, PriorNet

    model = CVAE(latent_dim=latent).to(device).eval()
    prior = PriorNet(y_dim=1, latent_dim=latent).to(device).eval()

    model_path = PROJECT_ROOT / "checkpoints" / "model_weights_trans_TransCVAE_PSMILES.pth"
    prior_path = PROJECT_ROOT / "checkpoints" / "model_weights_prior_trans_TransCVAE_PSMILES.pth"

    model.load_state_dict(torch.load(model_path, map_location=device))
    prior.load_state_dict(torch.load(prior_path, map_location=device))
    return model, prior


In [ ]:
save_weights()
